# Lesson 29 Lab — Custom Kernels: Packing, Dequantization, and CUTLASS Boundaries

**Puzzle:** When is an INT4 pack/dequant kernel worth building instead of using an existing backend?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

A custom INT4 kernel earns its complexity only if it removes work from the end-to-end path. Packing weights is helpful, but materializing a full dequantized matrix before calling BF16 GEMM adds reads, writes, conversions, and launches. The target is a fused load–unpack–scale–MMA–epilogue path with a supported tile layout.


## 0. Predict before running

1. Calculate packed bytes for a 4096×4096 INT4 weight matrix.
2. Predict the latency of a composed unpack/dequant/GEMM path relative to direct BF16 GEMM at M=32.
3. Name the evidence needed before calling an implementation a CUTLASS or custom-kernel result.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

An INT4 execution path contains pack/storage, scale loads, unpack/dequant, GEMM, epilogue, launches, and integration with framework layouts and streams.

- End-to-end gain includes unpack, scale loads, dequantization, GEMM, launch overhead, and integration cost.
- A Python or composed PyTorch prototype validates semantics but is not a fused CUTLASS kernel.
- The target shape distribution determines whether specialization pays off.


## 2. Derive the mechanism

The end-to-end budget is `T = T_pack/load + T_dequant + T_gemm + T_epilogue + overhead`. Fusing stages can remove intermediate traffic; a composed PyTorch reference intentionally exposes that unfused cost.

The logical pipeline is packed global load → nibble extraction/sign extension → scale load → dequantized fragments → matrix multiply/accumulate → epilogue. If dequantization writes a full BF16 matrix to global memory, the path pays both packed reads and a large materialized write/read before GEMM. Fusion keeps reconstructed values in registers/fragments and amortizes scale work across a tile.

Kernel profitability depends on M, N, K, group size, memory coalescing, register pressure, occupancy, and epilogue fusion. A semantic PyTorch composition is a correctness baseline and an upper-bound warning, not a custom kernel.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "29-custom-int4-kernels"
device = require_cuda()
torch.manual_seed(2026 + 29)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | direct BF16 GEMM for shape M=32, K=N=4096 |
| Candidate | PyTorch-composed unpack, sign restore, dequantization, and GEMM |
| Held constant | same X/W values, group size 128, packed layout, GPU timing helper |
| Measurements | packed bytes, BF16 median/p90, composed median/p90, implementation identity |
| Evidence | `pytorch-gpu` |

**Experiment:** Validate vectorized INT4 nibble packing/unpacking and time the composed PyTorch dequantize-plus-matmul path against BF16.


## 5. Read the experiment code

The lab validates nibble semantics and times a composed unpack-dequant-matmul reference, labeling it explicitly as non-fused and non-CUTLASS.

The notebook packs two codes per byte, reconstructs signed codes, applies block scales, materializes BF16 weights, and multiplies. It times the complete composed function rather than timing only the final GEMM. The result field explicitly says `not fused CUTLASS`.

This gives a readable semantic reference for testing a future CUDA/Triton/CUTLASS implementation. The future kernel must match its outputs while eliminating materialization and reducing launches.

Only after these variables match the protocol should the cell be executed.


In [2]:
m,k,n=32,4096,4096; x=torch.randn(m,k,device=device,dtype=torch.bfloat16); w=torch.randn(n,k,device=device,dtype=torch.bfloat16); q,scales,_=symmetric_quantize(w,bits=4,group_size=128)
codes=(q.to(torch.int16)&0xF).flatten(); packed=(codes[0::2]|(codes[1::2]<<4)).to(torch.uint8)
def composed():
    lo=(packed.to(torch.int16)&15); hi=((packed.to(torch.int16)>>4)&15); u=torch.stack([lo,hi],1).flatten(); u=torch.where(u>=8,u-16,u).reshape(n,k).float()
    dq=(u.reshape(n,k//128,128)*scales[...,None]).reshape(n,k).bfloat16(); return x@dq.t()
packed_t=cuda_benchmark(composed,warmup=3,repeats=10); bf16_t=cuda_benchmark(lambda:x@w.t(),warmup=5,repeats=15)
result=base_result(29,"pytorch-gpu"); result.update({"shape_mkn":[m,k,n],"packed_bytes":packed.numel(),"bf16_timing":bf16_t,"composed_unpack_dequant_matmul":packed_t,
    "implementation":"composed PyTorch reference, not fused CUTLASS","conclusion":"The composed reference exposed integration overhead; it is a semantic baseline, not a custom-kernel performance claim."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Shape M×K×N | 32 × 4096 × 4096 |
| Packed code bytes | 8,388,608 bytes |
| BF16 median | 0.027136 ms |
| Composed path median | 0.328720 ms |
| Implementation | composed PyTorch reference, not fused CUTLASS |


## 7. Interpret rather than merely print

Packed storage was 8,388,608 bytes for 16,777,216 weights, exactly 0.5 byte per code before scales. Direct BF16 GEMM took 0.027136 ms median. The composed unpack/dequant/matmul path took 0.328720 ms—about 12.1x slower.

The slowdown is not evidence that INT4 hardware is slow. It is evidence that the unfused reference performs too much integration work and memory traffic. It establishes the optimization target and a correctness oracle.

**Inspection rule:** Use the result to locate overhead, not to claim CUTLASS performance. A custom-kernel project begins only after a measured gap and stable shapes.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "bf16_timing": {
    "median_ms": 0.027136,
    "p90_ms": 0.027488,
    "repeats": 15,
    "samples_ms": [
      0.027136,
      0.028192,
      0.027424,
      0.027456,
      0.027136,
      0.027488,
      0.027136,
      0.02704,
      0.027072,
      0.026848,
      0.02704,
      0.026912,
      0.026112,
      0.027904,
      0.026432
    ],
    "warmup": 5
  },
  "composed_unpack_dequant_matmul": {
    "median_ms": 0.32872,
    "p90_ms": 0.330656,
    "repeats": 10,
    "samples_ms": [
      0.330656,
      0.330304,
      0.32256,
      0.330848,
      0.325088,
      0.32976,
      0.322944,
      0.330496,
      0.322944,
      0.32768
    ],
    "warmup": 3
  },
  "conclusion": "The composed reference exposed integration overhead; it is a semantic baseline, not a custom-kernel performance claim.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
 

## 9. Make the bounded decision

> Build custom code when the existing backend misses an important, repeated shape and the recoverable end-to-end budget exceeds integration cost.

**Acceptance/rollback:** First locate a repeated shape-level gap, verify pack/dequant semantics, profile roofline and memory traffic, implement, then require end-to-end gain and quality across the target shape distribution.

**Failure analysis:** Timing only GEMM after pre-dequantizing outside the measurement hides the dominant cost. Calling a Python composition a custom kernel is false. A fused kernel can also regress if register pressure lowers occupancy or if unsupported shapes fall back, so shape coverage and dispatch must be audited.


## 10. Extend the evidence

Implement a minimal fused kernel in CUTLASS, CUDA, or Triton for one frozen shape. Verify packed-layout compatibility and numerical parity, then profile instruction mix, global bytes, occupancy, tensor-pipe utilization, and end-to-end latency across M values. Add a safe fallback for unsupported shapes.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
